In [3]:
import pandas as pd

In [5]:
df = pd.read_csv("Dataset/raw/DSE_Data.csv")
df.head()

,Trading_Code,Date,Open,High,Low,Close,Volume
0,SIBL,2025-04-08,9.9,10.0,9.8,9.9,359773.0
1,MEGHNALIFE,2025-04-08,54.5,56.6,54.1,56.1,610711.0
2,BPML,2025-04-08,41.1,41.7,40.0,40.2,482031.0
3,CROWNCEMNT,2025-04-08,47.9,50.9,47.5,50.0,104860.0
4,ATCSLGF,2025-04-08,0.0,0.0,0.0,7.5,0.0


# duplicates


In [10]:
# Preserve original row number
df = df.reset_index().rename(columns={"index": "Row_Number"})

# Find all duplicates based on Trading_Code and Date
duplicates = df[
    df.duplicated(
        subset=["Trading_Code", "Date"],
        keep=False
    )
].sort_values(["Trading_Code", "Date"])

print(f"Total duplicate rows: {len(duplicates)}")
print(f"Duplicate groups: {duplicates.groupby(['Trading_Code','Date']).ngroups}")

duplicates.head(20)

Total duplicate rows: 132641
Duplicate groups: 63902


,Row_Number,Trading_Code,Date,Open,High,Low,Close,Volume
1129121,1129121,1JANATAMF,2012-01-10,8.0,8.8,8.0,8.6,1536500.0
1129126,1129126,1JANATAMF,2012-01-10,8.0,8.8,8.0,8.6,1536500.0
330519,330519,1JANATAMF,2022-07-18,6.3,6.3,6.2,6.3,313622.0
331038,331038,1JANATAMF,2022-07-18,6.3,6.3,6.2,6.3,313622.0
329753,329753,1JANATAMF,2022-07-19,6.2,6.2,6.2,6.2,241252.0
330136,330136,1JANATAMF,2022-07-19,6.2,6.2,6.2,6.2,241252.0
328965,328965,1JANATAMF,2022-07-20,6.1,6.3,6.1,6.2,132657.0
329372,329372,1JANATAMF,2022-07-20,6.1,6.3,6.1,6.2,132657.0
328367,328367,1JANATAMF,2022-07-21,6.2,6.3,6.2,6.2,279669.0
328748,328748,1JANATAMF,2022-07-21,6.2,6.3,6.2,6.2,279669.0


In [11]:
duplicates.to_csv(
    "duplicate_tradingcode_date_records.csv",
    index=False
)

print("Saved: duplicate_tradingcode_date_records.csv")

Saved: duplicate_tradingcode_date_records.csv


In [12]:
duplicate_summary = (
    duplicates
    .groupby(["Trading_Code", "Date"])
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=False)
)

duplicate_summary.head(20)

,Trading_Code,Date,Count
39056,NAVANAPHAR,2022-10-19,4
23836,GIB,2022-11-20,4
23839,GIB,2022-11-23,4
23838,GIB,2022-11-22,4
23837,GIB,2022-11-21,4
23835,GIB,2022-11-17,4
37637,MONNOAGML,2022-12-01,3
21431,FBFIF,2022-12-01,3
28453,IMAMBUTTON,2022-09-05,3
5619,APEXTANRY,2022-11-01,3


In [13]:
df_clean = df.drop_duplicates(
    subset=["Trading_Code", "Date"],
    keep="first"
)

In [14]:
print(f"Original rows: {len(df):,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Rows removed: {len(df)-len(df_clean):,}")

Original rows: 1,523,921
Cleaned rows: 1,455,182
Rows removed: 68,739


In [17]:
df_clean.head()

,Row_Number,Trading_Code,Date,Open,High,Low,Close,Volume
0,0,SIBL,2025-04-08,9.9,10.0,9.8,9.9,359773.0
1,1,MEGHNALIFE,2025-04-08,54.5,56.6,54.1,56.1,610711.0
2,2,BPML,2025-04-08,41.1,41.7,40.0,40.2,482031.0
3,3,CROWNCEMNT,2025-04-08,47.9,50.9,47.5,50.0,104860.0
4,4,ATCSLGF,2025-04-08,0.0,0.0,0.0,7.5,0.0


In [16]:
output_file = "Dataset/modified/DSE_Data_No_Duplicates.csv"

df_clean.to_csv(output_file, index=False)

print(f"Saved cleaned dataset to: {output_file}")

Saved cleaned dataset to: Dataset/modified/DSE_Data_No_Duplicates.csv


In [18]:
df = pd.read_csv('Dataset/modified/DSE_Data_No_Duplicates.csv')

In [19]:
df.head()

,Row_Number,Trading_Code,Date,Open,High,Low,Close,Volume
0,0,SIBL,2025-04-08,9.9,10.0,9.8,9.9,359773.0
1,1,MEGHNALIFE,2025-04-08,54.5,56.6,54.1,56.1,610711.0
2,2,BPML,2025-04-08,41.1,41.7,40.0,40.2,482031.0
3,3,CROWNCEMNT,2025-04-08,47.9,50.9,47.5,50.0,104860.0
4,4,ATCSLGF,2025-04-08,0.0,0.0,0.0,7.5,0.0


# Null value

In [20]:
df.isnull().sum()

Row_Number      0
Trading_Code    0
Date            0
Open            0
High            0
Low             0
Close           0
Volume          2
dtype: int64

In [21]:
df = df.dropna(subset=['Volume'])

In [22]:
df.to_csv("Dataset/modified/DSE_Data_Manupulation.csv", index=False)

# Sector Wise Data Marge

## Unique sector value

In [48]:
sector_df = pd.read_csv("Dataset/raw/sector_info/finance.csv")
sector_df.head()

,Symbol,Sector,Year,Eps,PE,Asst/Shr,Profit,Divid%,Div.Yield
0,1JANATAMF,MutualFunds,2023,-0.32,NaN,9.78,-92.55,NaN,NaN
1,1JANATAMF,MutualFunds,2022,0.24,26.47,10.80,70.10,7.0,10.94
2,1JANATAMF,MutualFunds,2021,2.54,2.76,11.94,735.56,13.0,18.57
3,1JANATAMF,MutualFunds,2020,-1.26,-3.25,9.32,-365.78,NaN,NaN
4,1JANATAMF,MutualFunds,2019,0.28,16.99,10.88,81.92,3.0,6.25


In [49]:
duplicates_sector = sector_df[
    sector_df.duplicated(
        subset=["Symbol", "Sector"],
        keep=False
    )
]

print(f"Total duplicate rows: {len(duplicates_sector)}")
print(f"Duplicate groups: {duplicates_sector.groupby(['Symbol','Sector']).ngroups}")

duplicates_sector.head(20)

Total duplicate rows: 7714
Duplicate groups: 464


,Symbol,Sector,Year,Eps,PE,Asst/Shr,Profit,Divid%,Div.Yield
0,1JANATAMF,MutualFunds,2023,-0.32,NaN,9.78,-92.55,NaN,NaN
1,1JANATAMF,MutualFunds,2022,0.24,26.47,10.80,70.10,7.0,10.94
2,1JANATAMF,MutualFunds,2021,2.54,2.76,11.94,735.56,13.0,18.57
3,1JANATAMF,MutualFunds,2020,-1.26,-3.25,9.32,-365.78,NaN,NaN
4,1JANATAMF,MutualFunds,2019,0.28,16.99,10.88,81.92,3.0,6.25
5,1JANATAMF,MutualFunds,2018,0.84,6.20,11.36,231.58,2.0,3.23
6,1JANATAMF,MutualFunds,2017,1.16,6.27,12.06,297.18,2.0,2.74
7,1JANATAMF,MutualFunds,2016,0.55,9.09,11.19,133.42,NaN,NaN
8,1JANATAMF,MutualFunds,2015,1.04,4.62,11.33,231.29,NaN,NaN
9,1JANATAMF,MutualFunds,2014,1.76,3.99,11.43,351.59,NaN,NaN


In [50]:
sector_df_clean = sector_df.drop_duplicates(
    subset=["Symbol", "Sector"],
    keep="first"
)
print(f"Original rows: {len(sector_df):,}")
print(f"Cleaned rows: {len(sector_df_clean):,}")
sector_df_clean = sector_df_clean[['Symbol', 'Sector']].rename(columns={'Symbol': 'trading_code'})
sector_df_clean.to_csv("Dataset/modified/finance_sector_info.csv", index=False)

Original rows: 7,725
Cleaned rows: 475


In [51]:
sector_df_clean.head()

,trading_code,Sector
0,1JANATAMF,MutualFunds
13,1STICB,MutualFunds
29,1STPRIMFMF,MutualFunds
43,2NDICB,MutualFunds
59,3RDICB,MutualFunds


In [52]:

merged_df = pd.merge(sector_df_clean, df, left_on='trading_code', right_on='Trading_Code', how='right')
merged_df = merged_df.drop(columns=['trading_code'])
# Reorder the list to place 'Sector' exactly where you want it
merged_df = merged_df[['Row_Number', 'Trading_Code', 'Sector', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume']]

In [53]:
merged_df.head()


,Row_Number,Trading_Code,Sector,Date,Open,High,Low,Close,Volume
0,0,SIBL,Bank,2025-04-08,9.9,10.0,9.8,9.9,359773.0
1,1,MEGHNALIFE,Insurance,2025-04-08,54.5,56.6,54.1,56.1,610711.0
2,2,BPML,PaperPrint,2025-04-08,41.1,41.7,40.0,40.2,482031.0
3,3,CROWNCEMNT,NaN,2025-04-08,47.9,50.9,47.5,50.0,104860.0
4,4,ATCSLGF,MutualFunds,2025-04-08,0.0,0.0,0.0,7.5,0.0


In [55]:
merged_df.to_csv("Dataset/modified/DSE_Data_Manupulation_with_Sector.csv", index=False)

In [62]:
unique_sectors = merged_df['Sector'].dropna().unique()
print("Unique sectors in the merged dataset:")
print(unique_sectors)

Unique sectors in the merged dataset:
['Bank' 'Insurance' 'PaperPrint' 'MutualFunds' 'Pharmaceuticals&Chemicals'
 'Miscellaneous' 'Textile' 'Telecommunication' 'Power&Fuel' 'Engineering'
 'Financial Institutions' 'Travel&Lesiure' 'ITSector' 'Food&Allied' 'Jute'
 'Services & Real Estate' 'Tannery Industries' 'Cement']


In [65]:
merged_df.isnull().sum()

Row_Number           0
Trading_Code         0
Sector          214307
Date                 0
Open                 0
High                 0
Low                  0
Close                0
Volume               0
dtype: int64